In [42]:
import os
import json
import pandas as pd

from langchain.schema import Document
from langchain.document_loaders import PyPDFLoader

In [43]:
documents = []

In [44]:
pdf_folder = "../data/pdfs"

for file in os.listdir(pdf_folder):

    if file.endswith(".pdf"):

        path = os.path.join(pdf_folder, file)

        loader = PyPDFLoader(path)

        pages = loader.load()

        for page in pages:

            department = file.split("_")[0]

            page.metadata["department"] = department
            page.metadata["source"] = file
            page.metadata["source_type"] = "pdf"

        documents.extend(pages)

print("PDFs loaded:", len(documents))

PDFs loaded: 4


In [45]:
documents[0]

Document(metadata={'source': 'compliance_security.pdf', 'page': 0, 'department': 'compliance', 'source_type': 'pdf'}, page_content='Enterprise  Compliance  &  Security   Sensitive  enterprise  data  must  follow  RBAC  policies.   Unauthorized  access  attempts  are  logged  and  audited.   Security  incidents  with  HIGH  severity  require  immediate  escalation.   Compliance  audits  are  performed  quarterly.    ')

In [46]:
csv_folder = "../data/csv"

for file in os.listdir(csv_folder):

    if file.endswith(".csv"):

        path = os.path.join(csv_folder, file)

        df = pd.read_csv(path)

        text = df.to_string(index=False)

        department = file.split("_")[0]

        doc = Document(
            page_content=text,
            metadata={
                "department": department,
                "source": file,
                "source_type": "csv"
            }
        )

        documents.append(doc)

print("Total docs after CSV:", len(documents))

Total docs after CSV: 7


In [47]:
json_folder = "../data/json"

for file in os.listdir(json_folder):

    if file.endswith(".json"):

        path = os.path.join(json_folder, file)

        with open(path, "r") as f:

            data = json.load(f)

        text = json.dumps(data, indent=2)

        doc = Document(
            page_content=text,
            metadata={
                "department": "logs",
                "source": file,
                "source_type": "json"
            }
        )

        documents.append(doc)

print("Final total documents:", len(documents))

Final total documents: 10


In [48]:
for doc in documents:
    print(doc.metadata)

{'source': 'compliance_security.pdf', 'page': 0, 'department': 'compliance', 'source_type': 'pdf'}
{'source': 'engineering_architecture.pdf', 'page': 0, 'department': 'engineering', 'source_type': 'pdf'}
{'source': 'finance_report.pdf', 'page': 0, 'department': 'finance', 'source_type': 'pdf'}
{'source': 'hr_policy.pdf', 'page': 0, 'department': 'hr', 'source_type': 'pdf'}
{'department': 'employees.csv', 'source': 'employees.csv', 'source_type': 'csv'}
{'department': 'finance', 'source': 'finance_transactions.csv', 'source_type': 'csv'}
{'department': 'operations', 'source': 'operations_inventory.csv', 'source_type': 'csv'}
{'department': 'logs', 'source': 'api_logs.json', 'source_type': 'json'}
{'department': 'logs', 'source': 'audit_logs.json', 'source_type': 'json'}
{'department': 'logs', 'source': 'security_alerts.json', 'source_type': 'json'}


In [49]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

In [50]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 10


In [51]:
chunks[0]

Document(metadata={'source': 'compliance_security.pdf', 'page': 0, 'department': 'compliance', 'source_type': 'pdf'}, page_content='Enterprise  Compliance  &  Security   Sensitive  enterprise  data  must  follow  RBAC  policies.   Unauthorized  access  attempts  are  logged  and  audited.   Security  incidents  with  HIGH  severity  require  immediate  escalation.   Compliance  audits  are  performed  quarterly.')

In [52]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [53]:
vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

print("Vector database created")

Vector database created


In [54]:
results = vectorstore.similarity_search(
    "What is the HR leave policy?",
    k=3
)

for doc in results:

    print("CONTENT:")
    print(doc.page_content)

    print("\nMETADATA:")
    print(doc.metadata)

    print("\n-------------------\n")

CONTENT:
HR  Leave  Policy   Employees  are  entitled  to  24  annual  paid  leave  days.   Remote  work  is  permitted  twice  per  week  with  manager  approval.   Employee  medical  insurance  is  provided  by  the  organization.   Confidential  employee  salary  data  must  only  be  accessed  by  HR  and  Finance  departments.

METADATA:
{'source': 'hr_policy.pdf', 'page': 0, 'department': 'hr', 'source_type': 'pdf'}

-------------------

CONTENT:
employee_id         name  department  salary  location
         101  John Carter Engineering   95000     Dubai
         102    Sarah Lee          HR   72000   Chennai
         103  Michael Tan     Finance  120000 Singapore
         104  David Kumar Engineering   98000 Bangalore
         105   Emily Wong     Finance  110000     Dubai
         106 Aisha Rahman          HR   76000 Abu Dhabi

METADATA:
{'department': 'employees.csv', 'source': 'employees.csv', 'source_type': 'csv'}

-------------------

CONTENT:
Enterprise  Compliance  &  Se

In [55]:
results = vectorstore.similarity_search(
    "financial compliance violations",
    k=3
)

for doc in results:
    print(doc.page_content)
    print("\n==========\n")

Finance  Audit  Report   Quarterly  operational  expenditure  increased  by  12%.   Cloud  infrastructure  spending  was  the  highest  operational  cost.   Unauthorized  financial  disclosure  is  considered  a  compliance  violation.   All  financial  records  are  restricted  to  Finance  department  personnel.


Enterprise  Compliance  &  Security   Sensitive  enterprise  data  must  follow  RBAC  policies.   Unauthorized  access  attempts  are  logged  and  audited.   Security  incidents  with  HIGH  severity  require  immediate  escalation.   Compliance  audits  are  performed  quarterly.


transaction_id  department  amount       vendor   status
        TXN001 Engineering   15000    CloudTech Approved
        TXN002     Finance   22000    AuditCorp  Pending
        TXN003          HR    5000   RecruitPro Approved
        TXN004 Engineering   45000 InfraSystems Approved
        TXN005     Finance   12000    SecurePay Rejected




Load Users & Access Policies

In [57]:
with open("../users.json", "r") as f:
    users = json.load(f)

with open("../access_policies.json", "r") as f:
    access_policies = json.load(f)

print(users)
print(access_policies)

{'alice': 'HR', 'bob': 'Finance', 'charlie': 'Engineering', 'admin': 'Admin'}
{'HR': ['hr'], 'Finance': ['finance'], 'Engineering': ['engineering'], 'Admin': ['hr', 'finance', 'engineering', 'logs', 'compliance']}


Function to Get Allowed Departments

In [58]:
def get_allowed_departments(username):

    role = users.get(username)

    if not role:
        return []

    allowed_departments = access_policies.get(role, [])

    return allowed_departments

Testing RBAC Mapping

In [59]:
print(get_allowed_departments("alice"))
print(get_allowed_departments("bob"))
print(get_allowed_departments("admin"))

['hr']
['finance']
['hr', 'finance', 'engineering', 'logs', 'compliance']


Secure Retrieval Function

In [60]:
def secure_retrieve(query, username, k=5):

    allowed_departments = get_allowed_departments(username)

    retrieved_docs = vectorstore.similarity_search(query, k=10)

    filtered_docs = []

    for doc in retrieved_docs:

        department = doc.metadata.get("department", "").lower()

        if department in allowed_departments:
            filtered_docs.append(doc)

    return filtered_docs[:k]

Testing Authorized Access

In [61]:
docs = secure_retrieve(
    "What is the leave policy?",
    "alice"
)

for doc in docs:

    print(doc.metadata)
    print(doc.page_content)

    print("\n=================\n")

{'source': 'hr_policy.pdf', 'page': 0, 'department': 'hr', 'source_type': 'pdf'}
HR  Leave  Policy   Employees  are  entitled  to  24  annual  paid  leave  days.   Remote  work  is  permitted  twice  per  week  with  manager  approval.   Employee  medical  insurance  is  provided  by  the  organization.   Confidential  employee  salary  data  must  only  be  accessed  by  HR  and  Finance  departments.




Testing Unauthorized Access

In [62]:
docs = secure_retrieve(
    "Show salary information",
    "alice"
)

print("Retrieved docs:", len(docs))

for doc in docs:
    print(doc.metadata)

Retrieved docs: 1
{'source': 'hr_policy.pdf', 'page': 0, 'department': 'hr', 'source_type': 'pdf'}


Testing Admin Access

In [63]:
docs = secure_retrieve(
    "security incidents and finance reports",
    "admin"
)

for doc in docs:
    print(doc.metadata)

{'source': 'finance_report.pdf', 'page': 0, 'department': 'finance', 'source_type': 'pdf'}
{'source': 'compliance_security.pdf', 'page': 0, 'department': 'compliance', 'source_type': 'pdf'}
{'department': 'finance', 'source': 'finance_transactions.csv', 'source_type': 'csv'}
{'department': 'logs', 'source': 'security_alerts.json', 'source_type': 'json'}
{'department': 'logs', 'source': 'audit_logs.json', 'source_type': 'json'}


In [64]:
import google.generativeai as genai

C:\Users\AditiRidhi\AppData\Local\Temp\ipykernel_12980\613638648.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
genai.configure(api_key="YOUR_API_KEY_HERE")

In [74]:
model = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini connected")

Gemini connected


In [75]:
response = model.generate_content(
    "Explain what RBAC means in one sentence."
)

print(response.text)

Role-Based Access Control (RBAC) is a method of regulating access to computer or network resources based on the individual's role within an organization, assigning permissions to roles rather than directly to users.


In [82]:
def generate_answer(query, retrieved_docs):

    context = "\n\n".join([
        doc.page_content for doc in retrieved_docs
    ])

    sources = list(set([
        doc.metadata.get("source", "unknown")
        for doc in retrieved_docs
    ]))

    prompt = f"""
You are a secure enterprise AI assistant.

Answer the user's question ONLY using the provided context.

Rules:
- Do NOT make up information
- If answer is not found, say:
  'I do not have enough authorized information.'
- Keep answer professional and concise
- Mention relevant enterprise details clearly

Context:
{context}

Question:
{query}
"""

    response = model.generate_content(prompt)

    answer = response.text

    answer += f"\n\nSources: {sources}"

    return answer, sources  

In [83]:
def ask_question(username, query):

    retrieved_docs = secure_retrieve(
        query=query,
        username=username,
        k=5
    )

    if len(retrieved_docs) == 0:

        return {
            "answer": "Access denied or no relevant authorized data found.",
            "sources": [],
            "confidence": 0
        }

    answer, sources = generate_answer(
        query,
        retrieved_docs
    )

    confidence = round(
        min(len(retrieved_docs) / 5, 1.0),
        2
    )

    return {
        "user": username,
        "query": query,
        "answer": answer,
        "sources": sources,
        "confidence": confidence
    }

In [88]:
result = ask_question(
    "alice",
    "What is the leave policy?"
)

print(result["answer"])


print("\nConfidence:", result["confidence"])

Employees are entitled to 24 annual paid leave days as per the HR Leave Policy.

Sources: ['hr_policy.pdf']

Confidence: 0.2


In [89]:
result = ask_question(
    "bob",
    "What are the financial audit findings?"
)

print(result["answer"])

Based on the Finance Audit Report, quarterly operational expenditure increased by 12%. Cloud infrastructure spending was identified as the highest operational cost.

Sources: ['finance_transactions.csv', 'finance_report.pdf']


In [90]:
result = ask_question(
    "alice",
    "Show financial transactions and audit findings"
)

print(result["answer"])

I do not have enough authorized information.

Sources: ['hr_policy.pdf']


In [91]:
result = ask_question(
    "admin",
    "Summarize security incidents and finance risks"
)

print(result["answer"])

print("\nConfidence:", result["confidence"])

**Security Incidents:**
A HIGH severity unauthorized payment access attempt was detected in the payment_api system. Additionally, multiple failed login attempts with MEDIUM severity occurred on the vpn_gateway. Security incidents with HIGH severity require immediate escalation, and unauthorized access attempts are logged and audited.

**Finance Risks:**
Quarterly operational expenditure increased by 12%, with cloud infrastructure spending identified as the highest operational cost. Unauthorized financial disclosure is considered a compliance violation. All financial records are restricted to Finance department personnel.

Sources: ['compliance_security.pdf', 'finance_transactions.csv', 'audit_logs.json', 'security_alerts.json', 'finance_report.pdf']

Confidence: 1.0
